# S03.4 – Мини-модель 1R (одно правило) против Dummy на честном протоколе

Цель: реализовать простейшую обучаемую модель своими руками и сравнить её с Dummy, не изучая "теорию моделей".

## Что вы освоите
- Понять интерфейс обучения как процедуру выбора правила по train
- Научиться реализовывать 1R: один признак + один порог
- Сравнить 1R с Dummy на одном split и по одним метрикам
- Сделать вывод: есть ли сигнал в данных для такого простого правила

## Важные оговорки
- Это учебная модель. Она нужна, чтобы студенты почувствовали механику обучения без математики.
- Важен именно честный протокол: правило выбираем только по train.

---


## Среда, воспроизводимость и стиль эксперимента

Перед кодом – несколько правил, которые будут повторяться во всех ноутбуках:

1) **Воспроизводимость**: фиксируем `random_state` / seed.  
2) **Разделение данных**: test‑часть – это *священная зона*. Мы смотрим на неё только в конце.  
3) **Всё, что "обучается" (`.fit`)** должно видеть только train‑данные (иначе легко получить утечки).  
4) **Sanity‑checks**: после каждого шага проверяем, что получился ожидаемый результат (формы, распределения, пересечения и т.д.).


In [1]:
# Импорты: минимальный, но достаточный набор
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Для красивых картинок (простая визуализация)
import matplotlib.pyplot as plt

# Фиксируем seed для воспроизводимости
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("numpy:", np.__version__)
print("pandas:", pd.__version__)


numpy: 2.0.2
pandas: 2.2.2


## Общие функции (оценка моделей и печать метрик)

Чтобы не копировать одно и то же вручную, заведём пару функций.

Важно: эти функции *ничего не делают магически*. Мы специально пишем их максимально прозрачно,
чтобы вы видели, какие именно метрики считаются и на каких данных.


In [2]:
def summarize_binary_metrics(y_true, y_pred, *, positive_label=1):
    """Считает базовые метрики бинарной классификации.

    Мы считаем:
    - accuracy: доля верных ответов
    - precision: насколько "чистые" наши позитивные предсказания
    - recall: насколько хорошо мы находим позитивный класс
    - f1: гармоническое среднее precision и recall

    Почему это важно: в задачах безопасности цена FP и FN может быть разной.
    """
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
    }

def print_confusion(y_true, y_pred, labels=(0,1)):
    """Печать матрицы ошибок и пояснения, что есть что."""
    cm = confusion_matrix(y_true, y_pred, labels=list(labels))
    df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
    display(df)
    tn, fp, fn, tp = cm.ravel()
    print(f"TN={tn} (верно: 0), FP={fp} (ложная тревога), FN={fn} (пропуск), TP={tp} (верно: 1)")
    return cm

def evaluate_model(model, X_train, y_train, X_test, y_test, *, model_name="model"):
    """Обучает модель на train и оценивает на test. Возвращает словарь метрик."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    metrics = summarize_binary_metrics(y_test, y_pred)
    print(f"=== {model_name} ===")
    print(metrics)
    print("Confusion matrix:")
    _ = print_confusion(y_test, y_pred)
    return metrics


## Почему 1R полезна в обучении

1R (one-rule) – это "классификатор одним правилом":
- выбираем один признак `j`,
- выбираем порог `t`,
- и говорим: `X[j] >= t → класс 1`, иначе 0 (или наоборот).

Такую модель легко объяснить:
- она почти как "if" в коде,
- её можно реализовать без библиотек,
- она всё равно требует честного train/test разделения.

Дальше мы реализуем её как класс с методами `fit/predict` (как в sklearn),
чтобы студенты привыкали к единому интерфейсу.


## Данные

Возьмём `breast_cancer`. Для простоты используем только часть признаков.


In [3]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)
print(X_train.shape, X_test.shape)


(426, 30) (143, 30)


## Реализация 1R

Алгоритм `fit`:
1) Перебираем признаки `j`.
2) Для каждого признака перебираем разумный набор порогов (например, квантили train).
3) Для каждого порога считаем ошибку на train.
4) Берём пару (j, t), дающую минимальную ошибку.

Это грубо, но прозрачно. И достаточно для учебных целей.


In [4]:
class OneRuleThresholdClassifier:
    """1R: один признак + один порог.

    Правило:
      если X[:, j] >= t → pred = 1, иначе 0
    или наоборот (мы выберем направление по train).
    """
    def __init__(self, n_thresholds=50):
        self.n_thresholds = n_thresholds
        self.feature_index_ = None
        self.threshold_ = None
        self.direction_ = None  # 1 означает: >= threshold => 1; -1 означает наоборот

    def fit(self, X, y):
        Xv = np.asarray(X)
        yv = np.asarray(y).astype(int)

        n_samples, n_features = Xv.shape
        best_err = float("inf")

        # Перебираем признаки
        for j in range(n_features):
            col = Xv[:, j]
            # Берём пороги как квантили (чтобы не перебрать миллионы уникальных значений)
            qs = np.linspace(0.02, 0.98, self.n_thresholds)
            thresholds = np.quantile(col, qs)

            for t in thresholds:
                # направление 1: >= t => 1
                pred1 = (col >= t).astype(int)
                err1 = (pred1 != yv).mean()

                # направление -1: < t => 1  (эквивалентно >= t => 0)
                pred2 = (col < t).astype(int)
                err2 = (pred2 != yv).mean()

                if err1 < best_err:
                    best_err = err1
                    self.feature_index_ = j
                    self.threshold_ = float(t)
                    self.direction_ = 1

                if err2 < best_err:
                    best_err = err2
                    self.feature_index_ = j
                    self.threshold_ = float(t)
                    self.direction_ = -1

        return self

    def predict(self, X):
        Xv = np.asarray(X)
        col = Xv[:, self.feature_index_]

        if self.direction_ == 1:
            return (col >= self.threshold_).astype(int)
        else:
            return (col < self.threshold_).astype(int)

    def describe_rule(self, feature_names=None):
        if feature_names is None:
            name = f"feature[{self.feature_index_}]"
        else:
            name = feature_names[self.feature_index_]
        if self.direction_ == 1:
            return f"IF {name} >= {self.threshold_:.4f} THEN 1 ELSE 0"
        else:
            return f"IF {name} < {self.threshold_:.4f} THEN 1 ELSE 0"


## Обучаем 1R только на train

Это принципиально: мы подбираем правило *по train*.  
Test оставляем для честной оценки.


In [5]:
oneR = OneRuleThresholdClassifier(n_thresholds=60)
oneR.fit(X_train.values, y_train.values)

rule = oneR.describe_rule(feature_names=list(X_train.columns))
print("Выученное правило:")
print(rule)


Выученное правило:
IF worst perimeter < 110.4390 THEN 1 ELSE 0


## Оцениваем на test и сравниваем с Dummy

Сначала – Dummy, потом 1R. Всё на одном и том же split.


In [6]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
y_dummy = dummy.predict(X_test)

y_oneR = oneR.predict(X_test.values)

print("Dummy metrics:", summarize_binary_metrics(y_test, y_dummy))
print_confusion(y_test, y_dummy)

print("1R metrics:", summarize_binary_metrics(y_test, y_oneR))
print_confusion(y_test, y_oneR)


Dummy metrics: {'accuracy': 0.6293706293706294, 'precision': 0.6293706293706294, 'recall': 1.0, 'f1': 0.7725321888412017}


,pred_0,pred_1
true_0,0,53
true_1,0,90


TN=0 (верно: 0), FP=53 (ложная тревога), FN=0 (пропуск), TP=90 (верно: 1)
1R metrics: {'accuracy': 0.916083916083916, 'precision': 0.9239130434782609, 'recall': 0.9444444444444444, 'f1': 0.9340659340659341}


,pred_0,pred_1
true_0,46,7
true_1,5,85


TN=46 (верно: 0), FP=7 (ложная тревога), FN=5 (пропуск), TP=85 (верно: 1)


array([[46,  7],
       [ 5, 85]])

## Интерпретация результата

Ответьте текстом (как преподаватель/студент) на вопросы:

1) 1R лучше Dummy? По какой метрике и на сколько?  
2) Что важнее в этой задаче – пропуски (FN) или ложные тревоги (FP)?  
3) Посмотрев на матрицу ошибок, какое поведение у 1R? Где она "ошибается"?  
4) Какой вывод о данных можно сделать: "есть сигнал" или "сигнал слабый"?

Важно: **не** делайте выводов уровня "эта модель лучшая". Здесь это просто тренировка протокола.


## Мини-итог S03.3

- Вы реализовали обучаемую модель без теории оптимизации.  
- Вы увидели, как важно разделять train/test.  
- Вы сравнили с Dummy и сделали интерпретацию.

В S05/S06 вы замените 1R на настоящие классификаторы, но каркас останется тот же.
